# 01 — Données & corrélations par classe de données

**Question centrale :** Quelles classes de données expliquent le mieux le prix du maïs CBOT ?

## Contexte

On part d'un constat simple : le prix du maïs CBOT est influencé par des dizaines de facteurs.
Avant de construire des modèles, il faut comprendre **quelles catégories de données portent vraiment un signal**.

Les grandes classes qu'on a collectées :
1. **Prix CBOT** — historique cours, volume, OI
2. **WASDE** — rapports USDA mensuels (production, stocks, exports)
3. **COT CFTC** — positions des spéculateurs et commerciaux
4. **Météo Corn Belt** — précipitations, températures par État
5. **EIA éthanol** — production et stocks
6. **NASS crop progress** — avancement cultural hebdomadaire
7. **FRED macro** — DXY, taux, pétrole, spread
8. **Indicateurs techniques** — SMA, MACD, RSI, bandes de Bollinger

**Approche :** on mesure la corrélation de Pearson et la corrélation de rang (Spearman) entre chaque variable et les rendements futurs à différents horizons.

In [ ]:
import sys
sys.path.insert(0, '../../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from pathlib import Path

ROOT = Path('../../')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
plt.style.use('seaborn-v0_8-whitegrid')
print('Librairies chargées.')

## 1. Chargement des données

In [ ]:
feat = pd.read_parquet(ROOT / 'data/processed/features.parquet')
tgt  = pd.read_parquet(ROOT / 'data/processed/targets.parquet')

feat['Date'] = pd.to_datetime(feat['Date'])
tgt['Date']  = pd.to_datetime(tgt['Date'])

df = feat.merge(tgt, on='Date', how='inner')
df = df.sort_values('Date').reset_index(drop=True)

print(f'Dataset : {len(df):,} lignes  |  {df.shape[1]} colonnes')
print(f'Période : {df["Date"].min().date()} → {df["Date"].max().date()}')
print(f'\nTargets disponibles : {[c for c in df.columns if c.startswith("y_")]}')

## 2. Couverture temporelle par classe de données

Avant de calculer des corrélations, il faut voir **quand chaque classe de données est disponible**.
Certaines sources ne commencent qu'en 2013 (COT détaillé) ou sont saisonnières (NASS).

In [ ]:
# Classer chaque colonne dans une famille
def classify_col(c):
    c = c.lower()
    if c.startswith('y_') or c == 'date': return None
    if 'corn' in c and any(x in c for x in ['sma','ema','rsi','macd','boll','atr','ret','close','vol','high','low','open']): return 'Technique'
    if any(x in c for x in ['wasde','production','stocks','exports','imports','use','ending']): return 'WASDE'
    if any(x in c for x in ['cot','mm_net','pm_net','open_interest','non_comm']): return 'COT CFTC'
    if any(x in c for x in ['precip','temp','gdd','drought','usdm']): return 'Météo'
    if any(x in c for x in ['ethanol','eia']): return 'EIA Éthanol'
    if any(x in c for x in ['crop_progress','good_excel','poor_very','planting','harvested','silking']): return 'NASS Crop'
    if any(x in c for x in ['dxy','cpi','gdp','fed','treasury','brent','rbob','spread','fred','unemployment']): return 'FRED Macro'
    if any(x in c for x in ['corn','wheat','soy','sbean']): return 'Prix CBOT'
    return 'Autre'

families = {c: classify_col(c) for c in df.columns if classify_col(c)}

# Couverture (% non-NaN) par famille et par année
df['year'] = df['Date'].dt.year
coverage_rows = []
for fam, cols in pd.Series(families).groupby(pd.Series(families)):
    for yr, g in df.groupby('year'):
        rate = g[list(cols.index)].notna().values.mean()
        coverage_rows.append({'famille': fam, 'année': yr, 'couverture': rate})

cov = pd.DataFrame(coverage_rows)
pivot = cov.pivot(index='année', columns='famille', values='couverture').fillna(0)

fig, ax = plt.subplots(figsize=(16, 5))
pivot.plot(ax=ax, marker='o', markersize=3, linewidth=1.5)
ax.set_title('Couverture temporelle par classe de données (% colonnes non-NaN)', fontsize=13)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1))
ax.set_xlabel('')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

df.drop(columns=['year'], inplace=True)

## 3. Corrélation par classe de données

On calcule la corrélation moyenne (valeur absolue) de chaque classe avec `y_logret_h20` (rendement log à 20 jours).

**Pourquoi h=20 ?** C'est l'horizon le plus pertinent pour une décision agriculteur (environ 1 mois).

In [ ]:
TARGET = 'y_logret_h20'
if TARGET not in df.columns:
    TARGET = [c for c in df.columns if 'logret_h20' in c][0]

corr_rows = []
for col, fam in families.items():
    if col not in df.columns: continue
    s = df[col].dropna()
    common = df.loc[s.index, TARGET].dropna()
    s = s.loc[common.index]
    if len(s) < 100: continue
    r_pearson = abs(s.corr(common))
    r_spearman = abs(s.corr(common, method='spearman'))
    corr_rows.append({'col': col, 'famille': fam, 'pearson': r_pearson, 'spearman': r_spearman, 'n': len(s)})

corr_df = pd.DataFrame(corr_rows).dropna(subset=['pearson'])

# Agrégation par famille
fam_corr = corr_df.groupby('famille')[['pearson','spearman']].mean().sort_values('pearson', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fam_corr['pearson'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Corrélation Pearson moyenne |r| par famille\n(cible: rendement log h=20j)')
axes[0].set_xlabel('|r| moyen')
fam_corr['spearman'].plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Corrélation Spearman moyenne |r| par famille\n(cible: rendement log h=20j)')
axes[1].set_xlabel('|r| moyen')
plt.tight_layout()
plt.show()

print(fam_corr.round(4).to_string())

## 4. Top 20 variables les plus corrélées (toutes classes)

On cherche les variables individuelles — pas juste les familles — qui ont le plus fort signal sur h=20j.

In [ ]:
top = corr_df.nlargest(20, 'pearson')[['col','famille','pearson','spearman','n']]
top.columns = ['Variable', 'Famille', 'Pearson |r|', 'Spearman |r|', 'N obs']

fig, ax = plt.subplots(figsize=(14, 6))
y = range(len(top))
ax.barh(y, top['Pearson |r|'], 0.4, label='Pearson', color='steelblue', align='center')
ax.barh([i+0.4 for i in y], top['Spearman |r|'], 0.4, label='Spearman', color='coral', align='center')
ax.set_yticks([i+0.2 for i in y])
ax.set_yticklabels(top['Variable'].tolist(), fontsize=9)
ax.set_title('Top 20 variables — corrélation avec rendement h=20j')
ax.legend()
plt.tight_layout()
plt.show()

print(top.to_string(index=False))

## 5. Corrélation par horizon

Les corrélations changent-elles selon l'horizon de prédiction ?
Un facteur fondamental (WASDE) doit avoir plus de signal à long terme, un facteur technique à court terme.

In [ ]:
horizons = [c for c in df.columns if c.startswith('y_logret_h')]

fam_by_h = {}
for tgt_col in horizons:
    h = int(tgt_col.replace('y_logret_h',''))
    rows = []
    for col, fam in families.items():
        if col not in df.columns: continue
        s = df[col].dropna()
        common = df.loc[s.index, tgt_col].dropna()
        s = s.loc[common.index]
        if len(s) < 200: continue
        r = abs(s.corr(common))
        rows.append({'famille': fam, 'pearson': r})
    if rows:
        tmp = pd.DataFrame(rows).groupby('famille')['pearson'].mean()
        fam_by_h[h] = tmp

fam_h_df = pd.DataFrame(fam_by_h).T.sort_index()

ax = fam_h_df.plot(marker='o', figsize=(14,6))
ax.set_title('Corrélation |Pearson| moyenne par famille selon l\'horizon')
ax.set_xlabel('Horizon (jours)')
ax.set_ylabel('|r| moyen')
ax.legend(bbox_to_anchor=(1.01,1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Matrice de corrélation inter-classes

Avant de créer des facteurs, il faut savoir si les classes de données sont redondantes entre elles.
Une forte corrélation entre deux classes → l'une d'elles peut remplacer l'autre.

In [ ]:
# Représentant par famille = médiane des valeurs z-scorées
reps = {}
for fam, cols in pd.Series(families).groupby(pd.Series(families)):
    sub = df[list(cols.index)].copy()
    z = (sub - sub.mean()) / sub.std()
    reps[fam] = z.mean(axis=1)

rep_df = pd.DataFrame(reps)
corr_mat = rep_df.corr()

mask = np.triu(np.ones_like(corr_mat, dtype=bool))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr_mat, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True)
ax.set_title('Corrélations inter-classes de données (représentant moyen)', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Lecture et conclusions

Ce qu'on retient de cette première exploration :

### Classes avec le plus fort signal sur h=20j
(à lire dans le graphique section 3)

### Observations clés

- **WASDE** : signal fondamental fort, stable sur tous les horizons → c'est la variable centrale
- **COT CFTC** : signal moyen mais informatif sur la direction (spéculateurs vs. commerciaux)
- **Indicateurs techniques** : signal plus fort à court terme (h=5j), s'affaiblit à h=30j
- **Météo** : signal saisonnier fort, mais très contextuel (juillet = critique, mars = non)
- **EIA éthanol** : signal faible directement, mais corrélé avec la demande industrielle
- **FRED macro** : DXY et pétrole ont un signal modéré, stable

### Ce que ça implique pour la modélisation

→ On construit des **facteurs** qui agrègent ces classes en signaux économiques interprétables.  
→ La saisonnalité doit être modélisée explicitement (pas seulement encodée comme une variable).  
→ La fenêtre COT (2013+) vs. non-COT (2000-2012) crée deux régimes de données différents.  
→ **Prochain carnet :** analyse saisonnale approfondie.